In [16]:
import ast
import re
import pandas as pd

import itertools

import numpy as np
import pandas as pd

from scipy.stats import kruskal, wilcoxon
from statsmodels.stats.multitest import multipletests

from src.config import RESULTS_DIR


In [5]:

RESULTS = RESULTS_DIR / "results.txt"

records = []

with open(RESULTS, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()


        # [{'algorithm': 'DemDS', ...}, {'algorithm': 'NuMDS', ...}]
        if not line.startswith("[{"):
            continue

        try:
            data = ast.literal_eval(line)

            for result in data:
                records.append(result)

        except (ValueError, SyntaxError) as e:
            print(f"Erro ao interpretar linha:\n{line}")
            print(e)


df = pd.DataFrame(records)

In [6]:
df

,algorithm,result,time,reported_time,seed,status,instance,beta
0,DemDS,1416108,33.217310,3.0,1,ok,n-10000000_beta-2_1_g-1,2.1
1,NuMDS,1416108,39.824540,NaN,1,ok,n-10000000_beta-2_1_g-1,2.1
2,DemDS,1416108,31.291225,2.0,2,ok,n-10000000_beta-2_1_g-1,2.1
3,NuMDS,1416108,38.755943,NaN,2,ok,n-10000000_beta-2_1_g-1,2.1
4,DemDS,1416108,29.902811,1.0,3,ok,n-10000000_beta-2_1_g-1,2.1
...,...,...,...,...,...,...,...,...
495,NuMDS,543884,36.237656,NaN,3,ok,n-10000000_beta-3_0_g-10,3.0
496,DemDS,546335,76.004123,10.0,4,ok,n-10000000_beta-3_0_g-10,3.0
497,NuMDS,543884,35.140651,NaN,4,ok,n-10000000_beta-3_0_g-10,3.0
498,DemDS,546171,76.758980,10.0,5,ok,n-10000000_beta-3_0_g-10,3.0


In [7]:
df["graph"] = (
    df["instance"]
    .str.extract(r"_g-(\d+)$")[0]
    .astype(int)
)


df = df.sort_values(
    ["beta", "graph", "algorithm", "seed"]
).reset_index(drop=True)

print("\nShape:", df.shape)
print("\nBetas:", sorted(df["beta"].unique()))
print("Algoritmos:", df["algorithm"].unique())
print("Grafos por beta:")
print(df.groupby("beta")["graph"].nunique())


Shape: (500, 9)

Betas: [np.float64(2.1), np.float64(2.4), np.float64(2.7), np.float64(3.0), np.float64(3.4)]
Algoritmos: <StringArray>
['DemDS', 'NuMDS']
Length: 2, dtype: str
Grafos por beta:
beta
2.1    10
2.4    10
2.7    10
3.0    10
3.4    10
Name: graph, dtype: int64


In [8]:
df

,algorithm,result,time,reported_time,seed,status,instance,beta,graph
0,DemDS,1416108,33.217310,3.0,1,ok,n-10000000_beta-2_1_g-1,2.1,1
1,DemDS,1416108,31.291225,2.0,2,ok,n-10000000_beta-2_1_g-1,2.1,1
2,DemDS,1416108,29.902811,1.0,3,ok,n-10000000_beta-2_1_g-1,2.1,1
3,DemDS,1416108,30.061988,1.0,4,ok,n-10000000_beta-2_1_g-1,2.1,1
4,DemDS,1416108,30.507408,2.0,5,ok,n-10000000_beta-2_1_g-1,2.1,1
...,...,...,...,...,...,...,...,...,...
495,NuMDS,584414,34.801329,NaN,1,ok,n-10000000_beta-3_4_g-10,3.4,10
496,NuMDS,584414,34.198178,NaN,2,ok,n-10000000_beta-3_4_g-10,3.4,10
497,NuMDS,584414,35.162612,NaN,3,ok,n-10000000_beta-3_4_g-10,3.4,10
498,NuMDS,584414,33.955191,NaN,4,ok,n-10000000_beta-3_4_g-10,3.4,10


In [9]:
# Median
df_median = (
    df
    .groupby(
        ["beta", "graph", "algorithm"],
        as_index=False
    )
    .agg(
        median_result=("result", "median"),
        n_seeds=("result", "count")
    )
)


print(df_median)


    beta  graph algorithm  median_result  n_seeds
0    2.1      1     DemDS      1416108.0        5
1    2.1      1     NuMDS      1416108.0        5
2    2.1      2     DemDS      1018014.0        5
3    2.1      2     NuMDS      1018014.0        5
4    2.1      3     DemDS      1279745.0        5
..   ...    ...       ...            ...      ...
95   3.4      8     NuMDS       584908.0        5
96   3.4      9     DemDS       591106.0        5
97   3.4      9     NuMDS       586481.0        5
98   3.4     10     DemDS       588977.0        5
99   3.4     10     NuMDS       584414.0        5

[100 rows x 5 columns]


In [10]:
df_comparison = (
    df_median
    .pivot(
        index=["beta", "graph"],
        columns="algorithm",
        values="median_result"
    )
    .reset_index()
)

df_comparison

algorithm,beta,graph,DemDS,NuMDS
0,2.1,1,1416108.0,1416108.0
1,2.1,2,1018014.0,1018014.0
2,2.1,3,1279745.0,1279745.0
3,2.1,4,842548.0,842548.0
4,2.1,5,1388765.0,1388765.0
5,2.1,6,1337147.0,1337147.0
6,2.1,7,1401043.0,1401043.0
7,2.1,8,1294583.0,1294583.0
8,2.1,9,1344232.0,1344232.0
9,2.1,10,1257295.0,1257295.0


In [11]:
# Relative difference between algorithms
# Obs: minimaztion problem
# delta = (DemDS - NuMDS)/NuMDS * 100
# 
#
# delta > 0: NuMDS better
# delta = 0: same result
# delta < 0: DemDS better


df_comparison["delta"] = (
    (df_comparison["DemDS"] - df_comparison["NuMDS"])
    / df_comparison["NuMDS"]
) * 100


print(df_comparison)

algorithm  beta  graph      DemDS      NuMDS     delta
0           2.1      1  1416108.0  1416108.0  0.000000
1           2.1      2  1018014.0  1018014.0  0.000000
2           2.1      3  1279745.0  1279745.0  0.000000
3           2.1      4   842548.0   842548.0  0.000000
4           2.1      5  1388765.0  1388765.0  0.000000
5           2.1      6  1337147.0  1337147.0  0.000000
6           2.1      7  1401043.0  1401043.0  0.000000
7           2.1      8  1294583.0  1294583.0  0.000000
8           2.1      9  1344232.0  1344232.0  0.000000
9           2.1     10  1257295.0  1257295.0  0.000000
10          2.4      1   614703.0   614651.0  0.008460
11          2.4      2   593827.0   593774.0  0.008926
12          2.4      3   608569.0   608522.0  0.007724
13          2.4      4   612354.0   612316.0  0.006206
14          2.4      5   607749.0   607690.0  0.009709
15          2.4      6   626656.0   626583.0  0.011650
16          2.4      7   619013.0   618950.0  0.010179
17        

In [ ]:
# Delta dataframe
df_deltas = (
    df_comparison
    .pivot(
        index="graph",
        columns="beta",
        values="delta"
    )
    .sort_index()
)

df_deltas.index.name = "graph"
df_deltas.columns.name = "beta"

df_deltas

beta,2.1,2.4,2.7,3.0,3.4
graph,,,,,
1,0.0,0.008460,0.150698,0.480649,0.781067
2,0.0,0.008926,0.140293,0.505051,0.793912
3,0.0,0.007724,0.144538,0.469445,0.775699
4,0.0,0.006206,0.132344,0.520279,0.791045
5,0.0,0.009709,0.140689,0.425322,0.795532
6,0.0,0.011650,0.131635,0.433944,0.790028
7,0.0,0.010179,0.127162,0.408290,0.765364
8,0.0,0.007435,0.121698,0.464573,0.792774
9,0.0,0.008395,0.140836,0.458993,0.788602


In [15]:
# dict
deltas = {
    beta: df_deltas[beta].dropna().tolist()
    for beta in df_deltas.columns
}

for beta, values in deltas.items():
    print(f"{beta}: {values}")

2.1: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
2.4: [0.008460085479402133, 0.008925954992977126, 0.0077236320133043665, 0.006205945949477068, 0.009708897628725174, 0.01165049163478741, 0.010178528152516358, 0.007435124497927055, 0.008395020271504832, 0.01175490683396698]
2.7: [0.1506980143652882, 0.14029300475405065, 0.14453815483125457, 0.13234445254587435, 0.140688997893447, 0.13163476299084834, 0.12716209223003064, 0.12169782753189569, 0.14083601772454366, 0.1279106290705287]
3.0: [0.48064939751416463, 0.5050514387125995, 0.46944479619158774, 0.5202788842416383, 0.4253221760102786, 0.43394442556837487, 0.4082900716208068, 0.46457329412479587, 0.45899305517137096, 0.45064756455420646]
3.4: [0.7810671109402366, 0.7939115199299167, 0.7756986402258562, 0.7910451840362949, 0.7955319159828198, 0.7900279698281462, 0.7653644828539917, 0.7927742482578456, 0.7886018472891705, 0.7807821167870722]


In [20]:
medianas_por_beta = {
    beta: np.median(valores)
    for beta, valores in deltas.items()
}

medianas_por_beta

{2.1: np.float64(0.0),
 2.4: np.float64(0.00869302023618963),
 2.7: np.float64(0.1363187286499625),
 3.0: np.float64(0.4617831746480834),
 3.4: np.float64(0.7893149085586584)}

In [ ]:
ALPHA = 0.05


# ============================================================
# Kruskal-Wallis
# ============================================================

groups = [
    df_deltas[beta].dropna().values
    for beta in df_deltas.columns
]

H, p_global = kruskal(*groups)

print("============================================================")
print("KRUSKAL-WALLIS GLOBAL")
print("============================================================")

print(f"H = {H:.6f}")
print(f"p-value = {p_global:.10g}")

if p_global < ALPHA:
    print("Resultado: rejeita H0.")
    print("Há evidência de diferença entre os betas.")
else:
    print("Resultado: não rejeita H0.")
    print("Não há evidência suficiente de diferença entre os betas.")

KRUSKAL-WALLIS GLOBAL
H = 47.434656
p-value = 1.237931956e-09
Resultado: rejeita H0.
Há evidência de diferença entre os betas.


In [ ]:

# ============================================================
# Mann-Whitney U.
# ============================================================

pairwise_beta = {}

raw_p_values = []
comparisons = []


for beta1, beta2 in itertools.combinations(df_deltas.columns, 2):

    x = df_deltas[beta1].dropna().values
    y = df_deltas[beta2].dropna().values

    from scipy.stats import mannwhitneyu

    statistic, p = mannwhitneyu(
        x,
        y,
        alternative="two-sided",
    )

    comparisons.append((beta1, beta2))
    raw_p_values.append(p)


# Holm
reject, corrected_p_values, _, _ = multipletests(
    raw_p_values,
    alpha=ALPHA,
    method="holm",
)


for (beta1, beta2), p_raw, p_corr, rejected in zip(
    comparisons,
    raw_p_values,
    corrected_p_values,
    reject,
):
    pairwise_beta[(beta1, beta2)] = {
        "p_raw": p_raw,
        "p_holm": p_corr,
        "reject_H0": bool(rejected),
    }


print("\n============================================================")
print("COMPARAÇÕES ENTRE BETAS")
print("============================================================")

for comparison, result in pairwise_beta.items():

    beta1, beta2 = comparison

    print(
        f"{beta1} vs {beta2}: "
        f"p={result['p_raw']:.6g}, "
        f"p_holm={result['p_holm']:.6g}, "
        f"reject={result['reject_H0']}"
    )


COMPARAÇÕES ENTRE BETAS
2.1 vs 2.4: p=6.38644e-05, p_holm=0.000638644, reject=True
2.1 vs 2.7: p=6.38644e-05, p_holm=0.000638644, reject=True
2.1 vs 3.0: p=6.38644e-05, p_holm=0.000638644, reject=True
2.1 vs 3.4: p=6.38644e-05, p_holm=0.000638644, reject=True
2.4 vs 2.7: p=0.000182672, p_holm=0.00109603, reject=True
2.4 vs 3.0: p=0.000182672, p_holm=0.00109603, reject=True
2.4 vs 3.4: p=0.000182672, p_holm=0.00109603, reject=True
2.7 vs 3.0: p=0.000182672, p_holm=0.00109603, reject=True
2.7 vs 3.4: p=0.000182672, p_holm=0.00109603, reject=True
3.0 vs 3.4: p=0.000182672, p_holm=0.00109603, reject=True
